In [1]:
# import the required libraries
import numpy as np
import cv2
import math
import csv

In [2]:
# a function that will resize the images so that they will fit 
# on screen in case they are big enough. 

def resize(img):
        return cv2.resize(img,(512,512)) # arg1- input image, arg- output_width, output_height

In [3]:
# saving data to csv

# open the file in the write mode
f = open('angle_len_data.csv', 'w')

# create the csv writer
writer = csv.writer(f)

header = ['n', 'angle', 'len1', 'len2']
# write the header
writer.writerow(header)

19

In [4]:
# We are using the HSV colour format because it is more sensitive to minor changes 
# in external lighting. Hence it will give more accurate masks and hence better results.
# filter out the red channel and create a mask frame.
# Red channel in hsv format is present in [0,230,170] to [255,255,220] range.
# l_b=np.array([0,230,170])# lower hsv bound for red
# u_b=np.array([255,255,220])# upper hsv bound to red

l_b=np.array([0,100,170])# lower hsv bound for red
u_b=np.array([255,255,220])# upper hsv bound to red

# Load image, grayscale, Otsu's threshold
image = cv2.imread('worm_actual_dots.jpg')
imageS = cv2.resize(image, (340, 560)) 

gray = cv2.cvtColor(imageS, cv2.COLOR_BGR2GRAY)
thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1] # cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

hsv=cv2.cvtColor(imageS,cv2.COLOR_BGR2HSV)
mask=cv2.inRange(hsv,l_b,u_b)

# Find circles 
cnts = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = cnts[0] if len(cnts) == 2 else cnts[1]
circles = []
center = []
for c in cnts:
    area = cv2.contourArea(c)
    if area > 10 and area < 500:
        ((x, y), r) = cv2.minEnclosingCircle(c)
        # Get center coordinates and radii of all circles (converted to int)
        circles.append([cv2.minEnclosingCircle(c)])
        circle = cv2.minEnclosingCircle(c) 
        center.append([np.int32(np.round(circle[0]))])
        cv2.circle(imageS, (int(x), int(y)), int(r), (36, 255, 12), 2)

# draw squares between circles

# Find nearest neighbours
centers = np.array(center)
n = len(centers)
tol = 20 # may have to increase tolerance when bending?

x_match = np.array([np.abs(centers[:,0,0] - cent[0,0]) < tol for cent in centers]) # vertical lines
y_match = np.array([np.abs(centers[:,0,1] - cent[0,1]) < tol for cent in centers]) # horizontal lines

# finding the shortest vertical lines so you don't get the ones that just go straight across  
track_dist = []
track_i = []
track_j = []

for i in np.arange(n):
    for j in np.arange(i+1,n):
        if (x_match[i, j]):
            track_dist.append(abs(centers[i,0,1] - centers[j,0,1]))
            track_i.append(i)
            track_j.append(j)

sort_dist = sorted(zip(track_dist,track_i,track_j))

# keep shortest 4 distances (4 will change for actual actuator)
# sort_dist = sort_dist[:-2]
for m in np.arange(10, len(sort_dist)):  
    x_match[sort_dist[m][1], sort_dist[m][2]] = False

# draw lines for squares on image
# use centre of horizontal lines for the spine
spine_coords = []
for i in np.arange(n): # i is centers[i]

    for j in np.arange(i+1, n):
        if (y_match[i, j]):
            start_hy = centers[i,0,1]
            end_hy = centers[j,0,1]
            start_hx = centers[i,0,0]
            end_hx = centers[j,0,0]
            cv2.line(imageS, (start_hx, start_hy), (end_hx, end_hy), (255, 0, 0), 2)

            # use centre of horizontal lines for the spine
            spine_coords.append([math.floor((start_hx + end_hx)/2), math.floor((start_hy + end_hy)/2)])
 
        if (x_match[i, j]):
            start_vx = centers[i,0,0]
            end_vx = centers[j,0,0]
            start_vy = centers[i,0,1]
            end_vy = centers[j,0,1]
            cv2.line(imageS, (start_vx, start_vy), (end_vx, end_vy), (255, 0, 0), 2)

    # draw spine
    for s in range(len(spine_coords)-1):
        cv2.line(imageS, (spine_coords[s][0],spine_coords[s][1]), (spine_coords[s+1][0],spine_coords[s+1][1]), (0, 0, 255), 2)
 
    for s in range(len(spine_coords)-2):
        p1x = spine_coords[s][0]
        p1y = spine_coords[s][1]

        p2x = spine_coords[s+1][0]
        p2y = spine_coords[s+1][1]

        p3x = spine_coords[s+2][0]
        p3y = spine_coords[s+2][1]

        dx1 = np.diff([p1x, p2x])
        dy1 = np.diff([p1y, p2y])

        dx2 = np.diff([p2x, p3x])
        dy2 = np.diff([p2y, p3y])

        m1 = dy1/dx1
        m2 = dy2/dx2

        angle = math.atan(abs((m1-m2)/(1+m1*m2))) * 180/math.pi        
        print("angle: ", angle)

cv2.imshow('thresh', thresh)
cv2.waitKey()
cv2.imshow('opening', mask)
cv2.waitKey()
cv2.imshow('image', imageS)
cv2.waitKey()
cv2.destroyAllWindows()

angle:  2.1427201644785763
angle:  2.1427201644785763
angle:  2.1427201644785763
angle:  3.9081270729406565
angle:  2.1427201644785763
angle:  3.9081270729406565
angle:  2.1427201644785763
angle:  3.9081270729406565
angle:  1.6545761863990338
angle:  2.1427201644785763
angle:  3.9081270729406565
angle:  1.6545761863990338
angle:  2.1427201644785763
angle:  3.9081270729406565
angle:  1.6545761863990338
angle:  0.5006994343924603
angle:  2.1427201644785763
angle:  3.9081270729406565
angle:  1.6545761863990338
angle:  0.5006994343924603


In [5]:
# find dots and draw squares between dots

# sets the video source to the default webcam, which OpenCV can easily capture.
video_capture = cv2.VideoCapture("worm_video1.mp4")

# We are using the HSV colour format because it is more sensitive to minor changes 
# in external lighting. Hence it will give more accurate masks and hence better results.
# filter out the red channel and create a mask frame.
# Red channel in hsv format is present in [0,230,170] to [255,255,220] range.
l_b=np.array([0,100,100])# lower hsv bound for red
u_b=np.array([255,255,220])# upper hsv bound to red

while True:
    # Capture frame-by-frame - read() function reads one frame from the video source, which in this example is the webcam. 
    # This returns: (1) actual video frame read (one frame on each loop) and (2) a return code (return code tells us if we 
    # have run out of frames, which will happen if we are reading from a file. This doesn’t matter when reading from the 
    # webcam, since we can record forever, so we will ignore it.
    ret, frame = video_capture.read()
    if (ret == False):
        break
    
    frame = cv2.resize(frame, (340, 560))

    hsv=cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
    mask=cv2.inRange(hsv,l_b,u_b)
    
    # Find circles 
    # findContours() detects changes in image colour and marks these as contours
    cnts = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    center = []
    for c in cnts:
        area = cv2.contourArea(c)

        # the area of the contour must be within the range expected for the dot drawn on the actuator.
        # If it is, a circle is drawn around the contour.
        if area > 10 and area < 1000:
            ((x, y), r) = cv2.minEnclosingCircle(c)
            circle = cv2.minEnclosingCircle(c) 
            center.append([np.int32(np.round(circle[0]))])
            cv2.circle(frame, (int(x), int(y)), int(r), (36, 255, 12), 2)

    # draw squares between circles to create a skeletal structure of the worm

    centers = np.array(center) # convert to numpy
    n = len(centers) # calculate number of circles 
    tol = 55 # may have to increase tolerance when bending?

    # compare coordinates of dots, if an x or y coordinate is similar mark it as True
    x_match = np.array([np.abs(centers[:,0,0] - cent[0,0]) < tol for cent in centers]) # vertical lines
    y_match = np.array([np.abs(centers[:,0,1] - cent[0,1]) < tol for cent in centers]) # horizontal lines

    # finding the shortest vertical lines so you don't get the ones that just go straight across  
    track_dist_v = []
    track_i_v = []
    track_j_v = []

    # create an ordered list of all the vertical lines in current frame
    for i in np.arange(n):
        for j in np.arange(i+1,n):
            if (x_match[i, j]):
                track_dist_v.append(abs(centers[i,0,1] - centers[j,0,1]))
                track_i_v.append(i)
                track_j_v.append(j)
    sort_dist_v = sorted(zip(track_dist_v,track_i_v,track_j_v))

    # keep shortest n distances (10 for actuator)
    for m in np.arange(10, len(sort_dist_v)):  
        x_match[sort_dist_v[m][1], sort_dist_v[m][2]] = False

    # finding the shortest horizontal lines so you don't get the diagonal ones  
    track_dist_h = []
    track_i_h = []
    track_j_h = []

    # create an ordered list of all the horizontal lines in current frame
    for i in np.arange(n):
        for j in np.arange(i+1,n):
            if (y_match[i, j]):
                track_dist_h.append(abs(centers[i,0,0] - centers[j,0,0]))
                track_i_h.append(i)
                track_j_h.append(j)
    sort_dist_h = sorted(zip(track_dist_h,track_i_h,track_j_h))
    track_dist_h = sorted(track_dist_h)

    # remove any of the tiny vertical lines
    for m in np.arange(len(sort_dist_h)):
        if track_dist_h[m] < 40:
            sort_dist_h[m] = (sort_dist_h[m][0] + 100, sort_dist_h[m][1], sort_dist_h[m][2])

    sort_dist_h = sorted(sort_dist_h)

    # keep shortest n distances (6 for actuator)
    for m in np.arange(6, len(sort_dist_h)):  
        y_match[sort_dist_h[m][1], sort_dist_h[m][2]] = False

    # draw lines for squares on image
    spine_coords = []
    for i in np.arange(n): # i is centers[i]
        for j in np.arange(i+1, n):
            # horizontal lines
            if (y_match[i, j]):
                start_hy = centers[i,0,1]
                end_hy = centers[j,0,1]
                start_hx = centers[i,0,0]
                end_hx = centers[j,0,0]
                cv2.line(frame, (start_hx, start_hy), (end_hx, end_hy), (255, 0, 0), 2)

                # use centre of horizontal lines for the spine
                spine_coords.append([math.floor((start_hx + end_hx)/2), math.floor((start_hy + end_hy)/2)])

            # vertical lines        
            if (x_match[i, j]):
                start_vx = centers[i,0,0]
                end_vx = centers[j,0,0]
                start_vy = centers[i,0,1]
                end_vy = centers[j,0,1]
                cv2.line(frame, (start_vx, start_vy), (end_vx, end_vy), (255, 0, 0), 2)

    # draw spine
    for s in range(len(spine_coords)-1):
        cv2.line(frame, (spine_coords[s][0],spine_coords[s][1]), (spine_coords[s+1][0],spine_coords[s+1][1]), (0, 0, 255), 2)

    for s in range(len(spine_coords)-2):
        p1x = spine_coords[s][0]
        p1y = spine_coords[s][1]

        p2x = spine_coords[s+1][0]
        p2y = spine_coords[s+1][1]

        p3x = spine_coords[s+2][0]
        p3y = spine_coords[s+2][1]

        dx1 = np.diff([p1x, p2x])
        dy1 = np.diff([p1y, p2y])

        dx2 = np.diff([p2x, p3x])
        dy2 = np.diff([p2y, p3y])

        len1 = math.sqrt( (p2x - p1x)**2 + (p2y - p1y)**2 )
        len2 = math.sqrt( (p3x - p2x)**2 + (p3y - p2y)**2 )

        m1 = dy1/dx1
        m2 = dy2/dx2

        angle = math.atan(abs((m1-m2)/(1+m1*m2))) * 180/math.pi        
        # print("angle: ", angle, s) 
        # print("length of lines", len1, len2)
        data = [s, angle, len1, len2]
        writer.writerow(data)
                              
    cv2.imshow('opening', mask)
    cv2.waitKey()
    # display video with respective lines and circles drawn on actuator.
    cv2.imshow('image', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# When everything is done, release the capture
video_capture.release()
cv2.destroyAllWindows()
f.close() # close the csv file

C:\Users\User\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:144: RuntimeWarning: divide by zero encountered in true_divide
C:\Users\User\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:146: RuntimeWarning: invalid value encountered in true_divide
C:\Users\User\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:143: RuntimeWarning: divide by zero encountered in true_divide
C:\Users\User\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:146: RuntimeWarning: invalid value encountered in subtract
